In [4]:
import torch
import lightning.pytorch as pl
import numpy as np
import os
from typing import Optional, Union
from tqdm import tqdm

from models.vqvae import VQVAE
from models.resnet import ResNet
from dataset.forecast_dataset import ForecastDataset


In [5]:
### Train ###

# Hyperparameters
lr = 5e-5
batch_size = 4
epochs = 3
device = "cuda" if torch.cuda.is_available() else "cpu"

# Data
path = "train.memmap"
channels = 5
img_res = (128, 64)

# Model

checkpoint_vqvae = "logs/vqvae_5channel/version_46/checkpoints/epoch=201-step=4751.ckpt"
checkpoint_resnet = "forecast_resnet.pth"

vqvae = VQVAE.load_from_checkpoint(checkpoint_vqvae, in_channels=channels).to(device)
vqvae.freeze()
resnet = ResNet(in_channels=8).to(device)

if os.path.exists(checkpoint_resnet):
    resnet.load_state_dict(torch.load(checkpoint_resnet))

# Dataset
dataset = ForecastDataset(path, vqvae, channels, img_res)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Optimizer
optimizer = torch.optim.Adam(resnet.parameters(), lr=lr)
criteria = torch.nn.functional.mse_loss


# Train Loop
for epoch in range(epochs):
    for x1, x2 in tqdm(dataloader):

        x1 = x1.to(device)
        x2 = x2.to(device)

        pred = resnet(x1)

        pred_quantized, quantize_loss, _ = vqvae.quantize(pred)

        prediction_loss = criteria(pred_quantized, x2)
        commitment_loss = quantize_loss["commitment_loss"]

        loss = prediction_loss + 0.1 * commitment_loss

        tqdm.write(f"Loss: {loss.item()}")

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# Save Model
torch.save(resnet.state_dict(), "forecast_resnet.pth")

C:\Users\hendr\AppData\Local\Temp\ipykernel_19508\3616090353.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  resnet.load_state_dict(torch.load(checkpoint_resnet))
  0%|

Loss: 0.03033062256872654


  1%|          | 2/200 [00:00<00:40,  4.83it/s]

Loss: 0.028589608147740364


  2%|▏         | 3/200 [00:00<00:41,  4.80it/s]

...

Loss: 0.02606368064880371


 99%|█████████▉| 198/200 [00:38<00:00,  5.16it/s]

Loss: 0.026279950514435768


100%|██████████| 200/200 [00:39<00:00,  5.12it/s]


Loss: 0.027742896229028702
Loss: 0.03061976283788681


In [9]:
sample = dataset[0][0]
sample = sample.unsqueeze(0)
print(sample.shape)

pred = resnet(sample)
pred = vqvae.quantize(pred)[0]
print(pred.shape)

mse = criteria(pred, dataset[0][1].unsqueeze(0))
print(mse)

torch.Size([1, 8, 16, 8])
torch.Size([1, 8, 16, 8])
tensor(0.0349, device='cuda:0', grad_fn=<MseLossBackward0>)
